# ResuMatch — Resume/JD Fit Classifier Training

Fine-tunes DistilBERT as a 3-class resume<->job-description fit classifier on the
[cnamuangtoun/resume-job-description-fit](https://huggingface.co/datasets/cnamuangtoun/resume-job-description-fit)
dataset (8,000 real resume-JD pairs), then exports to ONNX for the ResuMatch Streamlit app.

**Before running:** in the Kaggle notebook settings (right sidebar), turn on:
- **Accelerator: GPU T4 x2** (or any GPU)
- **Internet: On** (needed to download the dataset and the base DistilBERT weights)

Training runs on TensorFlow/Keras via **KerasHub** (the Keras team's own actively-maintained
model library) — not HuggingFace's TF model classes, which HuggingFace has deprecated and
removed as of `transformers` v5. `transformers` is still used, but only for its
framework-agnostic fast tokenizer. No GPU-capability fallback is needed here (unlike an
earlier PyTorch version of this pipeline) — TensorFlow's prebuilt pip wheels are compiled for
a broad compute-capability range (including Kaggle's older P100s), whereas recent PyTorch
wheels dropped compiled-kernel support for those older archs.

This notebook writes out the same scripts that live in `training/scripts/` in the ResuMatch
repo (kept in sync — copy-pasted verbatim, not reimplemented), so behavior matches what you'd
get running them locally. The export step (cell 4) works around two real KerasHub/tf2onnx
compatibility issues, verified by hand — see the comments in `export_model.py`'s docstring for
what and why.

Run all cells top to bottom. At the end, `resumatch_model.zip` in the notebook's Output tab
contains everything to drop into `ResuMatch/models/` locally.

In [ ]:
!pip install -q transformers keras-hub tf2onnx onnxruntime onnx

## 1. Prepare dataset

In [ ]:
%%writefile /kaggle/working/prepare_dataset.py
"""Download the resume-JD fit dataset from Hugging Face and carve out a val split.

Source: cnamuangtoun/resume-job-description-fit (6,241 train / 1,759 test rows,
columns: resume_text, job_description_text, label in {No Fit, Potential Fit, Good Fit}).
The published test.csv is kept untouched as the final holdout. val.csv is a GROUP split
by job_description_text (not a plain row-level stratified split) -- train.csv only has 279
unique JDs reused across 5,304 rows, so a naive random split put ~all of val's JDs into
train too (just paired with different resumes), inflating val accuracy on a task the model
had effectively already seen. Grouping by JD keeps val's job descriptions fully disjoint
from train's, matching how the official test.csv is already structured, so val now measures
genuine generalization to unseen JDs instead of "new resume, already-seen JD."

Usage (Kaggle notebook, internet enabled):
    python prepare_dataset.py --output_dir /kaggle/working/data
"""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import pandas as pd
from huggingface_hub import hf_hub_download
from sklearn.model_selection import GroupShuffleSplit

REPO_ID = "cnamuangtoun/resume-job-description-fit"

# Fixed order (not alphabetical) so the label is ordinal: index reflects fit strength.
LABELS = ["No Fit", "Potential Fit", "Good Fit"]


def download_csv(filename: str, retries: int = 5, backoff_seconds: float = 5.0) -> pd.DataFrame:
    """Uses huggingface_hub's resolution API rather than a raw GET against resolve/main/ --
    a raw GET against the CSV URL was seen to hit sustained 504s from Kaggle's network across
    every retry, while hf_hub_download hits HF's actual CDN resolution path and handles its
    own connection retries too."""
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            path = hf_hub_download(repo_id=REPO_ID, filename=filename, repo_type="dataset")
            return pd.read_csv(path)
        except Exception as e:
            last_error = e
            if attempt == retries:
                raise
            wait = backoff_seconds * (2 ** (attempt - 1))
            print(f"download failed ({e}), retrying in {wait:.0f}s ({attempt}/{retries})")
            time.sleep(wait)
    raise last_error  # unreachable, satisfies type checkers


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--output_dir", required=True, type=Path)
    parser.add_argument("--val_size", type=float, default=0.15)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    args.output_dir.mkdir(parents=True, exist_ok=True)

    full_train = download_csv("train.csv")
    test = download_csv("test.csv")
    print(f"downloaded: train={len(full_train)} test={len(test)}")

    assert set(full_train["label"].unique()) <= set(LABELS), "unexpected label values"

    splitter = GroupShuffleSplit(n_splits=1, test_size=args.val_size, random_state=args.seed)
    train_idx, val_idx = next(splitter.split(full_train, groups=full_train["job_description_text"]))
    train, val = full_train.iloc[train_idx], full_train.iloc[val_idx]

    train_jds = set(train["job_description_text"])
    val_jds = set(val["job_description_text"])
    overlap = train_jds & val_jds
    assert not overlap, f"group split failed: {len(overlap)} job descriptions leaked into both train and val"

    print(f"split: train={len(train)} val={len(val)} test={len(test)} "
          f"(train JDs={len(train_jds)}, val JDs={len(val_jds)}, overlap={len(overlap)})")

    train.to_csv(args.output_dir / "train.csv", index=False)
    val.to_csv(args.output_dir / "val.csv", index=False)
    test.to_csv(args.output_dir / "test.csv", index=False)

    with open(args.output_dir / "labels.json", "w") as f:
        json.dump(LABELS, f, indent=2)
    print(f"labels.json written: {LABELS}")


if __name__ == "__main__":
    main()


In [ ]:
!python /kaggle/working/prepare_dataset.py --output_dir /kaggle/working/data

## 2. Train

In [ ]:
%%writefile /kaggle/working/train_config.yaml
data_dir: /kaggle/working/data   # output of prepare_dataset.py: train.csv, val.csv, test.csv
run_name: resumefit_distilbert_v1
output_dir: /kaggle/working/runs

model_name: distilbert-base-uncased
resume_max_tokens: 350   # resumes front-load the most relevant info (summary, recent experience)
jd_max_tokens: 150       # so a fixed per-side budget beats naive concatenated truncation
max_seq_len: 512

batch_size: 16
num_workers: 2
epochs: 4

lr: 0.00002
warmup_ratio: 0.1
weight_decay: 0.01

early_stopping_patience: 2
seed: 42


In [ ]:
%%writefile /kaggle/working/train.py
"""Fine-tune DistilBERT as a 3-class resume<->job-description fit classifier.

Single-phase full fine-tune (unlike the image projects' freeze/unfreeze recipe --
transformer fine-tuning conventionally trains all layers at a low LR from the start).
Class-weighted loss handles the label imbalance (~50% No Fit / 25% / 25%).

TensorFlow/Keras via KerasHub (previously PyTorch via HuggingFace Transformers).
HuggingFace deprecated and is removing TensorFlow support from `transformers` (the TF model
classes are gone as of v5, and even the last v4.x release prints a deprecation warning and
has a broken PyTorch->TF weight-conversion path for this model) -- KerasHub is the Keras
team's own actively-maintained model library and isn't affected by that. `transformers` is
still used here, but only for its framework-agnostic fast tokenizer (no torch/TF pulled in
by that import). See export_model.py for a tf2onnx compatibility issue this pipeline also
had to work around.

Usage:
    python train.py --config ../configs/train_config.yaml
"""
from __future__ import annotations

import argparse
import json
import random
import time
from pathlib import Path

import keras
import keras_hub
import numpy as np
import pandas as pd
import tensorflow as tf
import yaml
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import DistilBertTokenizerFast

LABELS = ["No Fit", "Potential Fit", "Good Fit"]
KERAS_HUB_PRESET = "distil_bert_base_en_uncased"


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def configure_device() -> str:
    """Enable memory growth on any visible GPU. Unlike the PyTorch version this replaced,
    no compute-capability fallback is needed: TensorFlow's prebuilt pip wheels are compiled
    for a broad compute-capability range (including Kaggle's older P100s, sm_60), whereas
    recent PyTorch wheels dropped compiled-kernel support for those and needed an explicit
    CPU fallback to avoid crashing."""
    gpus = tf.config.list_physical_devices("GPU")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    device = "GPU" if gpus else "CPU"
    print(f"device: {device} ({len(gpus)} visible)")
    return device


def encode_pair(tokenizer, resume_text: str, jd_text: str, resume_max_tokens: int, jd_max_tokens: int) -> list[int]:
    """Tokenizes resume and JD separately with fixed per-side token budgets, then combines
    as [CLS] resume [SEP] jd [SEP] -- avoids the default pair-truncation strategy silently
    favoring whichever text happens to be shorter.
    """
    resume_ids = tokenizer.encode(
        resume_text, add_special_tokens=False, truncation=True, max_length=resume_max_tokens
    )
    jd_ids = tokenizer.encode(jd_text, add_special_tokens=False, truncation=True, max_length=jd_max_tokens)
    return [tokenizer.cls_token_id] + resume_ids + [tokenizer.sep_token_id] + jd_ids + [tokenizer.sep_token_id]


def make_dataset(df: pd.DataFrame, labels: list[int], tokenizer, resume_max_tokens: int, jd_max_tokens: int,
                  batch_size: int, shuffle: bool) -> tf.data.Dataset:
    """Dynamic per-batch padding (via padded_batch), matching the original collate_fn's
    behavior instead of padding every example to a fixed global max length. Yields
    {"token_ids", "padding_mask"} dicts -- KerasHub's DistilBertBackbone input names --
    instead of HF's {"input_ids", "attention_mask"}.
    """
    resumes = df["resume_text"].tolist()
    jds = df["job_description_text"].tolist()

    def generator():
        for resume, jd, label in zip(resumes, jds, labels):
            token_ids = encode_pair(tokenizer, resume, jd, resume_max_tokens, jd_max_tokens)
            yield {"token_ids": token_ids, "padding_mask": [1] * len(token_ids)}, label

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            {
                "token_ids": tf.TensorSpec(shape=(None,), dtype=tf.int32),
                "padding_mask": tf.TensorSpec(shape=(None,), dtype=tf.int32),
            },
            tf.TensorSpec(shape=(), dtype=tf.int32),
        ),
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=len(labels), seed=0, reshuffle_each_iteration=True)
    ds = ds.padded_batch(
        batch_size,
        padded_shapes=({"token_ids": [None], "padding_mask": [None]}, []),
        padding_values=({"token_ids": tf.cast(tokenizer.pad_token_id, tf.int32), "padding_mask": 0}, 0),
    )
    return ds.prefetch(tf.data.AUTOTUNE)


class LinearWarmupDecay(keras.optimizers.schedules.LearningRateSchedule):
    """Linear warmup to `peak_lr` over `warmup_steps`, then linear decay to 0 by
    `total_steps` -- matches the shape of HF's get_linear_schedule_with_warmup (used by the
    PyTorch version this replaced) without depending on `transformers`' TF-specific
    optimization helpers.
    """

    def __init__(self, peak_lr: float, warmup_steps: int, total_steps: int):
        super().__init__()
        self.peak_lr = peak_lr
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_steps = tf.cast(max(self.warmup_steps, 1), tf.float32)
        total_steps = tf.cast(max(self.total_steps, 1), tf.float32)
        warmup_lr = self.peak_lr * (step / warmup_steps)
        decay_progress = (step - warmup_steps) / tf.maximum(total_steps - warmup_steps, 1.0)
        decay_lr = self.peak_lr * tf.maximum(0.0, 1.0 - decay_progress)
        return tf.where(step < warmup_steps, warmup_lr, decay_lr)

    def get_config(self):
        return {"peak_lr": self.peak_lr, "warmup_steps": self.warmup_steps, "total_steps": self.total_steps}


def run_epoch(model, dataset, class_weights: tf.Tensor, optimizer, train: bool, log_every: int = 25):
    loss_fn = keras.losses.SparseCategoricalCrossentropy(
        from_logits=True, reduction="none"
    )
    total_loss, n = 0.0, 0
    all_labels, all_preds = [], []
    phase = "train" if train else "val"
    start = time.time()
    step = 0

    for step, (inputs, labels) in enumerate(dataset, start=1):
        with tf.GradientTape() as tape:
            logits = model(inputs, training=train)
            per_example_loss = loss_fn(labels, logits)
            sample_weights = tf.gather(class_weights, labels)
            loss = tf.reduce_mean(per_example_loss * sample_weights)
        if train:
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))

        batch_size = int(labels.shape[0])
        total_loss += float(loss) * batch_size
        n += batch_size
        all_labels.extend(labels.numpy())
        all_preds.extend(tf.argmax(logits, axis=1).numpy())

        if step % log_every == 0:
            elapsed = time.time() - start
            print(f"  [{phase}] batch {step} "
                  f"({elapsed:.1f}s elapsed, {elapsed / step:.2f}s/batch, running_loss={total_loss / n:.4f})",
                  flush=True)

    elapsed = time.time() - start
    print(f"  [{phase}] {step} batches in {elapsed:.1f}s")
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return total_loss / n, macro_f1


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True, type=Path)
    args = parser.parse_args()

    with open(args.config) as f:
        cfg = yaml.safe_load(f)

    set_seed(cfg["seed"])
    configure_device()

    data_dir = Path(cfg["data_dir"])
    train_df = pd.read_csv(data_dir / "train.csv")
    val_df = pd.read_csv(data_dir / "val.csv")
    train_labels = [LABELS.index(label) for label in train_df["label"]]
    val_labels = [LABELS.index(label) for label in val_df["label"]]

    tokenizer = DistilBertTokenizerFast.from_pretrained(cfg["model_name"])
    train_ds = make_dataset(train_df, train_labels, tokenizer, cfg["resume_max_tokens"], cfg["jd_max_tokens"],
                             cfg["batch_size"], shuffle=True)
    val_ds = make_dataset(val_df, val_labels, tokenizer, cfg["resume_max_tokens"], cfg["jd_max_tokens"],
                           cfg["batch_size"], shuffle=False)

    class_weights = compute_class_weight("balanced", classes=np.arange(len(LABELS)), y=train_labels)
    print(f"class weights: {dict(zip(LABELS, class_weights))}")
    class_weights_tensor = tf.constant(class_weights, dtype=tf.float32)

    model = keras_hub.models.DistilBertClassifier.from_preset(
        KERAS_HUB_PRESET, num_classes=len(LABELS), preprocessor=None
    )

    steps_per_epoch = -(-len(train_labels) // cfg["batch_size"])  # ceil division
    total_steps = steps_per_epoch * cfg["epochs"]
    lr_schedule = LinearWarmupDecay(
        peak_lr=cfg["lr"], warmup_steps=int(total_steps * cfg["warmup_ratio"]), total_steps=total_steps
    )
    optimizer = keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=cfg["weight_decay"])

    run_dir = Path(cfg["output_dir"]) / cfg["run_name"]
    ckpt_dir = run_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    with open(run_dir / "labels.json", "w") as f:
        json.dump(LABELS, f, indent=2)

    # Auto-resume: a Kaggle run can take hours and risks hitting the platform's session time
    # limit, so every epoch's full training state is checkpointed -- rerunning the same
    # command picks up where it left off instead of restarting from scratch. Unlike the
    # PyTorch version's single last_state.pt bundle, a TF checkpoint only captures
    # model+optimizer variables (not arbitrary Python state), so epoch/best-F1/history are
    # tracked in a companion JSON file saved in lockstep with each checkpoint.
    tf_ckpt_dir = ckpt_dir / "last_state"
    checkpoint = tf.train.Checkpoint(model=model, optimizer=optimizer)
    manager = tf.train.CheckpointManager(checkpoint, str(tf_ckpt_dir), max_to_keep=1)
    meta_path = ckpt_dir / "last_state_meta.json"

    if manager.latest_checkpoint and meta_path.exists():
        checkpoint.restore(manager.latest_checkpoint).expect_partial()
        with open(meta_path) as f:
            meta = json.load(f)
        start_epoch = meta["epoch"] + 1
        best_macro_f1 = meta["best_macro_f1"]
        epochs_without_improvement = meta["epochs_without_improvement"]
        history = meta["history"]
        print(f"resuming from {manager.latest_checkpoint}: starting at epoch {start_epoch} "
              f"(best_val_macro_f1={best_macro_f1:.4f} so far)")
    else:
        start_epoch = 1
        best_macro_f1 = -1.0
        epochs_without_improvement = 0
        history = []

    best_path = ckpt_dir / "best.keras"
    for epoch in range(start_epoch, cfg["epochs"] + 1):
        train_loss, train_f1 = run_epoch(model, train_ds, class_weights_tensor, optimizer, train=True)
        val_loss, val_f1 = run_epoch(model, val_ds, class_weights_tensor, optimizer, train=False)
        print(f"epoch {epoch}: train_loss={train_loss:.4f} train_macro_f1={train_f1:.4f} "
              f"val_loss={val_loss:.4f} val_macro_f1={val_f1:.4f}")
        history.append({"epoch": epoch, "train_loss": train_loss, "train_macro_f1": train_f1,
                         "val_loss": val_loss, "val_macro_f1": val_f1})

        if val_f1 > best_macro_f1:
            best_macro_f1 = val_f1
            epochs_without_improvement = 0
            model.save(best_path)
        else:
            epochs_without_improvement += 1

        manager.save()
        with open(meta_path, "w") as f:
            json.dump({
                "epoch": epoch,
                "best_macro_f1": best_macro_f1,
                "epochs_without_improvement": epochs_without_improvement,
                "history": history,
            }, f, indent=2)

        if epochs_without_improvement >= cfg["early_stopping_patience"]:
            print(f"early stopping at epoch {epoch} (best val_macro_f1={best_macro_f1:.4f})")
            break

    with open(run_dir / "metrics.json", "w") as f:
        json.dump({"history": history, "best_val_macro_f1": best_macro_f1}, f, indent=2)

    print(f"done. best val_macro_f1={best_macro_f1:.4f}. best checkpoint: {best_path}")


if __name__ == "__main__":
    main()

In [ ]:
!python /kaggle/working/train.py --config /kaggle/working/train_config.yaml

## 3. Evaluate on the held-out test split

Headline metric is **macro-F1** (fairer than accuracy given the ~50/25/25 label imbalance).

In [ ]:
%%writefile /kaggle/working/evaluate.py
"""Evaluate a trained resume-fit checkpoint against the held-out test split.

Usage:
    python evaluate.py --config ../configs/train_config.yaml \
        --checkpoint /kaggle/working/runs/resumefit_distilbert_v1/checkpoints/best.keras
"""
from __future__ import annotations

import argparse
from pathlib import Path

import keras
import pandas as pd
import tensorflow as tf
import yaml
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from transformers import DistilBertTokenizerFast

from train import LABELS, configure_device, make_dataset


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True, type=Path)
    parser.add_argument("--checkpoint", required=True, type=Path)
    parser.add_argument("--split", default="test", choices=["val", "test"])
    args = parser.parse_args()

    with open(args.config) as f:
        cfg = yaml.safe_load(f)

    configure_device()

    tokenizer = DistilBertTokenizerFast.from_pretrained(cfg["model_name"])
    model = keras.models.load_model(args.checkpoint)

    df = pd.read_csv(Path(cfg["data_dir"]) / f"{args.split}.csv")
    labels = [LABELS.index(label) for label in df["label"]]
    ds = make_dataset(df, labels, tokenizer, cfg["resume_max_tokens"], cfg["jd_max_tokens"],
                       cfg["batch_size"], shuffle=False)

    all_labels, all_preds = [], []
    for inputs, batch_labels in ds:
        logits = model(inputs, training=False)
        all_preds.extend(tf.argmax(logits, axis=1).numpy())
        all_labels.extend(batch_labels.numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    print(f"split={args.split} n={len(all_labels)} macro_f1={macro_f1:.4f}")
    print(classification_report(all_labels, all_preds, target_names=LABELS, zero_division=0))
    print("confusion matrix (rows=true, cols=pred):")
    print(pd.DataFrame(confusion_matrix(all_labels, all_preds), index=LABELS, columns=LABELS))


if __name__ == "__main__":
    main()

In [ ]:
!python /kaggle/working/evaluate.py --config /kaggle/working/train_config.yaml     --checkpoint /kaggle/working/runs/resumefit_distilbert_v1/checkpoints/best.keras --split test

## 4. Export to ONNX

Exports the model + a standalone `tokenizer.json` (loadable via the lightweight `tokenizers` package, no `transformers`/`tensorflow` needed at inference), quantizes to int8 (required to stay under GitHub's 100MB push limit), and verifies both steps before trusting the artifact.

In [ ]:
%%writefile /kaggle/working/export_model.py
"""Export a trained resume-fit checkpoint to ONNX for CPU inference, quantize it to int8,
and verify both steps.

The fp32 export is ~255MB -- over GitHub's 100MB hard push limit -- so int8 dynamic
quantization (~67MB) is a required step here, not an optional optimization. Also saves a
standalone tokenizer.json so the Streamlit app can tokenize with the lightweight
`tokenizers` package instead of pulling in full `transformers` + `tensorflow`.

Three real compatibility issues had to be worked around here (all verified by hand against
the installed tf2onnx==1.17.0 -- not assumptions):

1. KerasHub's DistilBERT computes its exact (erf-based) GELU in a way that TensorFlow
   compiles down to an `Erfc` op, which tf2onnx has no converter for. Worked around with a
   custom op handler that rewrites `Erfc(x)` as the mathematically exact `1 - Erf(x)` --
   ONNX has a native `Erf` op, so this is a lossless rewrite, not an approximation.
2. tf2onnx's public `convert.from_function()` API crashes (verified: a real interpreter
   segfault / MemoryError, not a clean exception) inside its own `remove_back_to_back`
   graph-optimizer pass on this model's graph, in this tf2onnx/protobuf version combo.
   Worked around by calling tf2onnx's lower-level pipeline directly (frozen-graph -> graph
   conversion -> model_proto) and skipping just that optimizer pass -- the exported graph is
   less aggressively optimized as a result, but verified numerically identical to the
   unquantized TF model (see the `max abs diff` check below) and still gets int8-quantized
   the same as before.
3. KerasHub's attention layers use `EinsumDense` (a single fused `Einsum` op covering
   reshape+matmul+reshape), which `onnxruntime.quantization.quantize_dynamic` doesn't
   recognize at all -- its dynamic-quantization registry only covers
   MatMul/Gather/Conv/LSTM/Attention/Transpose, not Einsum. Left as-is, that silently skips
   quantizing all 24 attention Q/K/V/output-projection weight matrices (54MB of the ~255MB
   fp32 model), producing a 111MB "quantized" file that still blows past GitHub's 100MB push
   limit. `quantize_einsum_weights()` below does the same dynamic int8 quantization by hand
   for exactly those nodes (symmetric per-tensor, QuantizeLinear/DequantizeLinear pair
   inserted before each Einsum) before handing off to quantize_dynamic for the rest --
   verified this drops the final size to ~69MB with <0.001 max softmax-probability drift.

Usage:
    python export_model.py \
        --checkpoint /kaggle/working/runs/resumefit_distilbert_v1/checkpoints/best.keras \
        --output ../../models/resume_fit_distilbert.onnx
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import keras
import numpy as np
import onnx
import onnxruntime as ort
import tensorflow as tf
import tf2onnx
from onnx import helper, numpy_helper
from onnxruntime.quantization import QuantType, quantize_dynamic
from tf2onnx import tf_loader, utils as tf2onnx_utils
from tf2onnx.convert import tensor_names_from_structed
from tf2onnx.tfonnx import process_tf_graph
from transformers import DistilBertTokenizerFast

LABELS = ["No Fit", "Potential Fit", "Good Fit"]


def softmax(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def erfc_to_erf_handler(ctx, node, name, args):
    """tf2onnx has no converter for TF's Erfc op (see module docstring). Rewrites
    Erfc(x) -> 1 - Erf(x), which ONNX's native Erf op (opset 9+) supports directly --
    exact, not an approximation."""
    x = node.input[0]
    shapes = node.output_shapes
    dtypes = node.output_dtypes
    output_name = node.output[0]
    one_const = ctx.make_const(tf2onnx_utils.make_name("erfc_one"), np.array(1.0, dtype=np.float32))
    erf_node = ctx.make_node("Erf", [x], op_name_scope=name)
    ctx.remove_node(name)
    ctx.make_node("Sub", [one_const.output[0], erf_node.output[0]], outputs=[output_name],
                  name=name, shapes=shapes, dtypes=dtypes)


def quantize_einsum_weights(model_path: Path, min_weight_size: int = 1000):
    """Symmetric per-tensor int8 dynamic quantization for Einsum nodes' constant weight
    inputs -- see point 3 in the module docstring for why quantize_dynamic can't do this.
    In-place: overwrites model_path.
    """
    model = onnx.load(str(model_path))
    graph = model.graph
    initializers = {init.name: init for init in graph.initializer}

    new_nodes = []
    replaced_init_names = set()
    n_quantized = 0
    for node in graph.node:
        if node.op_type != "Einsum":
            new_nodes.append(node)
            continue
        weight_idx = next((i for i, inp in enumerate(node.input) if inp in initializers), None)
        if weight_idx is None:
            new_nodes.append(node)
            continue

        weight_name = node.input[weight_idx]
        weight_arr = numpy_helper.to_array(initializers[weight_name])
        if weight_arr.dtype != np.float32 or weight_arr.size < min_weight_size:
            new_nodes.append(node)
            continue

        max_abs = np.abs(weight_arr).max()
        scale = float(max_abs / 127.0) if max_abs > 0 else 1.0
        quantized = np.clip(np.round(weight_arr / scale), -127, 127).astype(np.int8)

        quant_name = weight_name + "_i8"
        scale_name = weight_name + "_scale"
        dequant_name = weight_name + "_dq"
        graph.initializer.append(numpy_helper.from_array(quantized, name=quant_name))
        graph.initializer.append(numpy_helper.from_array(np.array(scale, dtype=np.float32), name=scale_name))
        replaced_init_names.add(weight_name)

        new_nodes.append(helper.make_node(
            "DequantizeLinear", [quant_name, scale_name], [dequant_name], name=weight_name + "_DQ"
        ))
        node.input[weight_idx] = dequant_name
        new_nodes.append(node)
        n_quantized += 1

    del graph.node[:]
    graph.node.extend(new_nodes)
    kept_initializers = [init for init in graph.initializer if init.name not in replaced_init_names]
    del graph.initializer[:]
    graph.initializer.extend(kept_initializers)

    onnx.save(model, str(model_path))
    print(f"quantized {n_quantized} einsum weight tensors")


def convert_to_onnx(serving_fn, input_signature, opset: int, output_path: Path):
    """Equivalent to tf2onnx.convert.from_function(), except it skips the graph-optimizer
    pass that crashes on this model (see module docstring point 2)."""
    concrete_func = serving_fn.get_concrete_function(*input_signature)
    input_names = [t.name for t in concrete_func.inputs if t.dtype != tf.dtypes.resource]
    output_names = [t.name for t in concrete_func.outputs if t.dtype != tf.dtypes.resource]
    tensors_to_rename = tensor_names_from_structed(concrete_func, input_names, output_names)

    with tf.device("/cpu:0"):
        frozen_graph = tf_loader.from_function(concrete_func, input_names, output_names, large_model=False)
        with tf.Graph().as_default() as tf_graph:
            tf.import_graph_def(frozen_graph, name="")
            g = process_tf_graph(
                tf_graph, continue_on_error=True, opset=opset,
                input_names=input_names, output_names=output_names,
                custom_op_handlers={"Erfc": (erfc_to_erf_handler, [])},
                tensors_to_rename=tensors_to_rename,
            )
            model_proto = g.make_model("resumatch")

    tf2onnx_utils.save_protobuf(str(output_path), model_proto)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", required=True, type=Path)
    parser.add_argument("--output", required=True, type=Path)
    parser.add_argument("--tokenizer_model_name", default="distilbert-base-uncased",
                        help="HF model id the tokenizer vocab matches (same vocab as the KerasHub preset).")
    args = parser.parse_args()

    tokenizer = DistilBertTokenizerFast.from_pretrained(args.tokenizer_model_name)
    model = keras.models.load_model(args.checkpoint)

    # int64, named input_ids/attention_mask: matches the ONNX graph the deployed app.py
    # already calls (session.run(["logits"], {"input_ids": ..., "attention_mask": ...}) with
    # np.int64 arrays) so this is a drop-in replacement -- app.py needs no changes.
    # KerasHub's model itself expects {"token_ids", "padding_mask"} as int32, so serving_fn
    # renames/casts at the boundary.
    dummy_len = 32
    dummy_input_ids = tf.random.uniform((1, dummy_len), minval=0, maxval=tokenizer.vocab_size, dtype=tf.int64)
    dummy_attention_mask = tf.ones((1, dummy_len), dtype=tf.int64)

    args.output.parent.mkdir(parents=True, exist_ok=True)
    fp32_path = args.output.with_suffix(".fp32.onnx")

    input_signature = [
        tf.TensorSpec((None, None), tf.int64, name="input_ids"),
        tf.TensorSpec((None, None), tf.int64, name="attention_mask"),
    ]

    @tf.function(input_signature=input_signature)
    def serving_fn(input_ids, attention_mask):
        token_ids = tf.cast(input_ids, tf.int32)
        padding_mask = tf.cast(attention_mask, tf.int32)
        return {"logits": model({"token_ids": token_ids, "padding_mask": padding_mask})}

    convert_to_onnx(serving_fn, input_signature, opset=17, output_path=fp32_path)
    print(f"exported to {fp32_path}")

    tf_out = model({
        "token_ids": tf.cast(dummy_input_ids, tf.int32),
        "padding_mask": tf.cast(dummy_attention_mask, tf.int32),
    }).numpy()

    sess = ort.InferenceSession(str(fp32_path), providers=["CPUExecutionProvider"])
    onnx_out = sess.run(["logits"], {
        "input_ids": dummy_input_ids.numpy(),
        "attention_mask": dummy_attention_mask.numpy(),
    })[0]

    max_diff = np.abs(tf_out - onnx_out).max()
    print(f"max abs diff in logits (tf vs onnx): {max_diff:.6f}")

    tf_prob = softmax(tf_out)
    onnx_prob = softmax(onnx_out)
    max_prob_diff = np.abs(tf_prob - onnx_prob).max()
    print(f"max abs diff in probability (tf vs onnx): {max_prob_diff:.6f}")
    assert max_prob_diff < 0.01, "ONNX export does not match TensorFlow output — do not trust this artifact"
    print("fp32 export verified OK")

    quantize_einsum_weights(fp32_path)
    quantize_dynamic(str(fp32_path), str(args.output), weight_type=QuantType.QInt8)
    fp32_size = fp32_path.stat().st_size / 1024 / 1024
    quant_size = args.output.stat().st_size / 1024 / 1024
    fp32_path.unlink()
    print(f"quantized {fp32_size:.1f}MB -> {quant_size:.1f}MB: {args.output}")

    quant_sess = ort.InferenceSession(str(args.output), providers=["CPUExecutionProvider"])
    quant_out = quant_sess.run(["logits"], {
        "input_ids": dummy_input_ids.numpy(),
        "attention_mask": dummy_attention_mask.numpy(),
    })[0]
    quant_prob_diff = np.abs(tf_prob - softmax(quant_out)).max()
    print(f"max abs diff in probability (tf fp32 vs quantized): {quant_prob_diff:.4f}")
    # Loose tolerance -- quantization is lossy by design (expect a few points of accuracy
    # loss, verified separately via evaluate.py against real data), this just catches a
    # broken/garbage quantized export, not fine-grained accuracy regression.
    assert quant_prob_diff < 0.5, "quantized model diverges too much from fp32 -- do not trust this artifact"
    print("quantized export verified OK")

    # Standalone tokenizer.json (loadable via the `tokenizers` package, no `transformers` needed).
    tokenizer_dst = args.output.parent / "tokenizer.json"
    tokenizer.backend_tokenizer.save(str(tokenizer_dst))
    print(f"saved tokenizer to {tokenizer_dst}")

    labels_dst = args.output.parent / "labels.json"
    with open(labels_dst, "w") as f:
        json.dump(LABELS, f, indent=2)
    print(f"saved labels to {labels_dst}")


if __name__ == "__main__":
    main()

In [ ]:
!python /kaggle/working/export_model.py     --checkpoint /kaggle/working/runs/resumefit_distilbert_v1/checkpoints/best.keras     --output /kaggle/working/export/resume_fit_distilbert.onnx

## 5. Package for download

Download `resumatch_model.zip` from this notebook's **Output** tab, unzip it into `ResuMatch/models/` locally (should contain `resume_fit_distilbert.onnx`, `tokenizer.json`, `labels.json`), then run `streamlit run app.py`.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/resumatch_model", "zip", "/kaggle/working/export")
print("packaged: /kaggle/working/resumatch_model.zip")